# Experiment Management with UAEF

## How Experiments Work

In UAEF, an **experiment** is a single evaluation run persisted to AWS:
- **DynamoDB**: One row per experiment with average scores, evaluation count, and a pointer to S3
- **S3**: Full evaluation results (all metric scores for every test case) as JSON

Every call to `evaluate()` or `batch_evaluate()` with `persist=True` creates a **new experiment**.
If you want to compare two agent configurations, you run two separate experiments and compare their DynamoDB rows.

This notebook demonstrates:
1. Running an evaluation with persistence (creates experiment in DynamoDB + S3)
2. Retrieving experiment results
3. Listing all experiments
4. Comparing two experiments

---
## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

from dotenv import load_dotenv
load_dotenv()

from uaef.api import evaluate, batch_evaluate
from uaef.models import AgentTrace, Message, MessageRole, GroundTruth
from uaef.storage import get_storage
from datetime import datetime
from uuid import uuid4

print("✓ UAEF imported successfully")

---
## 2. Configuration

Experiment persistence uses DynamoDB + S3. Configure via environment variables:

```bash
export AWS_REGION=us-east-1
export UAEF_DYNAMODB_TABLE=uaef-experiments
export UAEF_S3_BUCKET=uaef-results
export UAEF_S3_PREFIX=evaluations/
```

Or check the current config:

In [ ]:
from uaef.config import get_config

config = get_config()
print(f"AWS Region:      {config.aws.region}")
print(f"DynamoDB Table:  {config.storage.dynamodb_table_name}")
print(f"S3 Bucket:       {config.storage.s3_bucket}")
print(f"S3 Prefix:       {config.storage.s3_prefix}")
print(f"LLM Judge Model: {config.llm_judge.model_id}")

---
## 3. Run Evaluation with Persistence

When you pass `persist=True` to `evaluate()` or `batch_evaluate()`, UAEF:
1. Creates the DynamoDB table and S3 bucket if they don't exist
2. Runs the evaluation
3. Writes full results to S3 as JSON
4. Writes a summary row to DynamoDB with average scores
5. Returns the `EvaluationResult` with `experiment_id` populated

In [ ]:
# Create a simple trace for demonstration
trace = AgentTrace(
    trace_id=str(uuid4()),
    messages=[
        Message(role=MessageRole.USER, content="What is the capital of France?", timestamp=datetime.utcnow()),
        Message(role=MessageRole.ASSISTANT, content="The capital of France is Paris.", timestamp=datetime.utcnow()),
    ],
    tool_calls=[],
    session_id="experiment-demo",
)

ground_truth = GroundTruth(expected_output="Paris", expected_tool_calls=[])

# Evaluate WITH persistence — creates a new experiment
result = evaluate(
    trace=trace,
    ground_truth=ground_truth,
    persist=True,
    experiment_name="Demo Experiment - Single Query",
    experiment_objective="Demonstrate experiment persistence with a simple QA trace",
)

print(f"✓ Evaluation complete")
print(f"  Overall Score:   {result.overall_score:.2f}")
print(f"  Passed:          {result.passed}")
print(f"  Experiment ID:   {result.experiment_id}")

---
## 4. Batch Evaluation with Persistence

For batch evaluation, all test cases are grouped under a single experiment.

In [ ]:
# Create multiple traces
test_cases = [
    ("What is 2+2?", "2+2 equals 4.", "4"),
    ("Who wrote Romeo and Juliet?", "William Shakespeare wrote Romeo and Juliet.", "William Shakespeare"),
    ("What is the largest planet?", "Jupiter is the largest planet.", "Jupiter"),
]

traces = []
ground_truths = []

for question, answer, expected in test_cases:
    traces.append(AgentTrace(
        trace_id=str(uuid4()),
        messages=[
            Message(role=MessageRole.USER, content=question, timestamp=datetime.utcnow()),
            Message(role=MessageRole.ASSISTANT, content=answer, timestamp=datetime.utcnow()),
        ],
        tool_calls=[],
        session_id="batch-demo",
    ))
    ground_truths.append(GroundTruth(expected_output=expected, expected_tool_calls=[]))

# Batch evaluate with persistence
results = batch_evaluate(
    traces=traces,
    ground_truths=ground_truths,
    persist=True,
    experiment_name="Demo Experiment - Batch",
    experiment_objective="Demonstrate batch experiment with 3 QA test cases",
)

print(f"✓ Batch evaluation complete")
print(f"  Test cases:      {len(results)}")
print(f"  Experiment ID:   {results[0].experiment_id}")
for i, r in enumerate(results):
    status = "✓" if r.passed else "✗"
    print(f"  {status} Test {i+1}: {r.overall_score:.2f}")

avg = sum(r.overall_score for r in results) / len(results)
print(f"\n  Average Score: {avg:.2f}")

---
## 5. Retrieve Experiment from DynamoDB

Each experiment is stored as a single row in DynamoDB with:
- `experiment_id` (partition key)
- `experiment_name`
- `experiment_objective`
- `evaluation_count`
- `average_scores` (per-metric averages)
- `overall_average_score`
- `result_path` (S3 URL to full results)

In [ ]:
store = get_storage()

# Retrieve by experiment ID
exp_id = str(results[0].experiment_id)
experiment = store.get_experiment(exp_id)

print(f"Experiment from DynamoDB:")
print(f"  ID:              {experiment['experiment_id']}")
print(f"  Name:            {experiment['experiment_name']}")
print(f"  Objective:       {experiment['experiment_objective']}")
print(f"  Eval Count:      {experiment['evaluation_count']}")
print(f"  Overall Avg:     {float(experiment['overall_average_score']):.4f}")
print(f"  Result Path:     {experiment['result_path']}")
print(f"  Created:         {experiment['created_at']}")
print(f"\n  Average Scores:")
for metric, score in sorted(experiment['average_scores'].items()):
    print(f"    {metric}: {float(score):.4f}")

---
## 6. Retrieve Full Results from S3

The S3 file contains the complete `EvaluationResult` objects for every test case.

In [ ]:
full_results = store.get_full_results(exp_id)

print(f"Full results from S3:")
print(f"  Experiment:    {full_results['experiment_name']}")
print(f"  Evaluations:   {len(full_results['evaluations'])}")

# Inspect first evaluation
first_eval = full_results['evaluations'][0]
print(f"\n  First evaluation:")
print(f"    Overall Score: {first_eval['overall_score']}")
print(f"    Passed:        {first_eval['passed']}")
print(f"    Dimensions:    {len(first_eval['dimension_results'])}")
for dim in first_eval['dimension_results']:
    print(f"      {dim['dimension_name']}: {dim['aggregate_score']:.3f}")

---
## 7. List All Experiments

Scan DynamoDB to see all experiments in the account.

In [ ]:
all_experiments = store.list_experiments()

print(f"Total experiments: {len(all_experiments)}")
print()
for exp in all_experiments:
    print(f"  {exp['experiment_name']}")
    print(f"    ID:       {exp['experiment_id']}")
    print(f"    Score:    {float(exp.get('overall_average_score', 0)):.3f}")
    print(f"    Count:    {exp.get('evaluation_count', 0)}")
    print(f"    Created:  {exp.get('created_at', 'N/A')}")
    print()

---
## 8. Compare Two Experiments

To compare agent configurations, run two experiments and compare their average scores.
Each experiment is independent — one experiment per evaluation run.

In [ ]:
# Find two experiments to compare
if len(all_experiments) >= 2:
    exp_a = all_experiments[0]
    exp_b = all_experiments[1]

    print(f"Comparing:")
    print(f"  A: {exp_a['experiment_name']} (score: {float(exp_a.get('overall_average_score', 0)):.3f})")
    print(f"  B: {exp_b['experiment_name']} (score: {float(exp_b.get('overall_average_score', 0)):.3f})")
    print()

    # Compare metric by metric
    scores_a = {k: float(v) for k, v in exp_a.get('average_scores', {}).items()}
    scores_b = {k: float(v) for k, v in exp_b.get('average_scores', {}).items()}

    common_metrics = sorted(set(scores_a.keys()) & set(scores_b.keys()))

    print(f"{'Metric':<35} {'A':>8} {'B':>8} {'Delta':>8}")
    print("-" * 65)
    for metric in common_metrics:
        a_val = scores_a[metric]
        b_val = scores_b[metric]
        delta = b_val - a_val
        indicator = "⬆" if delta > 0.05 else ("⬇" if delta < -0.05 else " ")
        print(f"  {indicator} {metric:<32} {a_val:>7.3f} {b_val:>7.3f} {delta:>+7.3f}")
else:
    print("Need at least 2 experiments to compare. Run more evaluations with persist=True.")

---
## 9. Delete an Experiment

Remove an experiment from both DynamoDB and S3.

In [ ]:
# Uncomment to delete an experiment:
# store.delete_experiment(exp_id)
# print(f"✓ Deleted experiment {exp_id}")

print("Skipping delete (uncomment above to run)")

---
## Summary

| Action | Code |
|--------|------|
| Create experiment | `evaluate(trace, persist=True, experiment_name="...")` |
| Batch experiment | `batch_evaluate(traces, persist=True, experiment_name="...")` |
| Get experiment | `store.get_experiment(experiment_id)` |
| Find by name | `store.get_experiment_by_name(name)` |
| Get full results | `store.get_full_results(experiment_id)` |
| List all | `store.list_experiments()` |
| Delete | `store.delete_experiment(experiment_id)` |

Each `persist=True` call creates a new experiment. To compare configurations, run separate experiments and compare their `average_scores` from DynamoDB.